# Transportation & Logistics Analytics: Fixing OTIF at BharatLink Freight

**End-to-end analysis: data audit → cleaning → feature engineering → 10 golden questions → rule-based delay warning → Power BI star-schema export**

| | |
|---|---|
| **Business** | BharatLink Freight (fictional), a 3PL running full-truck-load road freight for automotive & industrial OEMs across India |
| **Data** | 3,585 GPS-tracked bookings (Dec 2019 – Aug 2020, plus a few outliers) · 36 customers · 190+ carriers · 38 vehicle types |
| **Role** | Logistics Data Analyst |
| **Analyst** | Pham Hong Hieu |

### The 10 golden questions
| # | Theme | Question |
|---|---|---|
| 1 | OTIF health | What is network OTIF and is it trending up or down by month? |
| 2 | Route bottlenecks | Which regions / routes have abnormal lead times, and why? |
| 3 | Supplier performance | Which carriers are best and worst? Should we exit any? |
| 4 | Delay root cause | Do some material types get delayed more often? |
| 5 | Load utilisation | How many trips run empty or under-loaded? |
| 6 | Anomalies | Which trips have GPS distance inconsistent with the straight-line distance? |
| 7 | Seasonality | Weekday vs. weekend performance: why is Wednesday often late? |
| 8 | Cost efficiency | How does freight cost per unit vary by region? |
| 9 | Breaking points | Which high-volume regions have low success: add a hub or change carriers? |
| 10 | Delay prediction | A rule-based early-warning model for customer service at booking time |

## Part I: Setup & Data Ingestion

In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

NAVY, ORANGE, TEAL, RED, GREEN, GREY = "#0F2A4A", "#F97316", "#0E7490", "#DC2626", "#16A34A", "#94A3B8"
pio.templates["bharatlink"] = go.layout.Template(layout=dict(
    font=dict(family="Segoe UI, Arial", size=13, color="#1F2937"), colorway=[NAVY, ORANGE, TEAL, RED, GREEN, GREY],
    plot_bgcolor="white", paper_bgcolor="white", xaxis=dict(showgrid=False, linecolor="#CBD5E1"),
    yaxis=dict(gridcolor="#E5E7EB"), title=dict(font=dict(size=17))))
pio.templates.default = "plotly_white+bharatlink"
pio.renderers.default = "plotly_mimetype+notebook_connected"

RAW = Path("../02_Data/raw/Transportation & Logistics Tracking Dataset.xlsx")
PROCESSED = Path("../02_Data/processed"); PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS = Path("outputs"); OUTPUTS.mkdir(exist_ok=True)

raw = pd.read_excel(RAW, sheet_name="Primary Data")
dictionary = pd.read_excel(RAW, sheet_name="Data Dictionary", header=None, names=["Field", "Description"])
print(raw.shape)
raw.head(3).T

(3585, 28)


,0,1,2
Gps Provider,Consent Track,Vamosys,Vamosys
Booking ID,AEIBK2027469,VCV00014153/082021,VCV00014063/082021
Shipment Type,Regular,Regular,Regular
Booking Date,2020-08-26 12:03:46,2020-08-27 15:21:48.570000,2020-08-27 14:22:17.833000
Vehicle Registration,MH14GD9464,TN30BC9320,TN30BB1036
Origin Location,"Shive, Pune, Maharashtra","Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,..."
Destination Location,"Pondur, Kanchipuram, Tamil Nadu","Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,..."
Origin Location Latitude,18.751,12.839,12.839
Origin Location Longitude,73.877,79.954,79.954
Destination Location Latitude,12.930,12.839,12.839


## Part II: Data Quality Audit

Unlike a curated teaching dataset, this is a **raw operational GPS export**, and it shows it. The audit follows the *Data Cleaning Check* checklist: column naming, mixed data types, placeholder values, missing data, duplicates, inconsistent text, invalid ranges and business-rule conflicts.

In [2]:
dq = []
def log(check, result, action, rows=None):
    dq.append({"Check": check, "Result": result, "Rows affected": rows, "Action": action})

# 1. Column naming: typo + inconsistent naming vs. the data dictionary
print([c for c in raw.columns if "Curren " in c])
log("Column naming", "Typo 'Curren Location Longitude'; names differ from dictionary", "Rename to clean PascalCase names")

# 2. Mixed data types inside date columns
date_cols = ["Booking Date", "Data Ping time", "Planned ETA", "Actual ETA", "Trip Start Date", "Trip End Date"]
for c in date_cols:
    types = raw[c].map(lambda v: type(v).__name__).value_counts().to_dict()
    print(f"{c:<18} {types}")
mixed = sum(raw[c].map(lambda v: isinstance(v, str)).sum() for c in date_cols)
log("Mixed types in date columns", "datetime + str + float in the same column", "pd.to_datetime(errors='coerce')", int(mixed))

['Curren Location Longitude']
Booking Date       {'Timestamp': 3585}
Data Ping time     {'Timestamp': 3584, 'NaTType': 1}
Planned ETA        {'datetime': 3583, 'str': 2}
Actual ETA         {'datetime': 3558, 'float': 25, 'str': 2}
Trip Start Date    {'datetime': 3583, 'str': 2}
Trip End Date      {'datetime': 3583, 'str': 2}


In [3]:
# 3. Placeholder values: Excel's zero-date 1899-12-30, 'Unknown' carriers, 'Xxxxxx' materials
parsed = {c: pd.to_datetime(raw[c], errors="coerce") for c in date_cols}
zero_dates = sum(int((parsed[c].dt.year < 1950).sum()) for c in date_cols)
unknown_sup = int((raw["Supplier Name"].str.strip().str.lower() == "unknown").sum())
x_mat = int(raw["Material Shipped"].str.fullmatch(r"(?i)x+").fillna(False).sum())
print("1899-12-30 placeholder dates:", zero_dates, "| 'Unknown' carrier:", unknown_sup, "| 'Xxxxxx' material:", x_mat)
log("Placeholder dates (1899-12-30)", "Excel zero-date used as 'blank'", "Set to NaT", zero_dates)
log("Placeholder carrier 'Unknown'", "Carrier not recorded", "Keep as 'Unknown carrier' (own category)", unknown_sup)
log("Placeholder material 'Xxxxxx'", "Material not recorded", "Map to 'Unspecified'", x_mat)

# 4. Missing values
miss = raw.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False)
display(pd.DataFrame({"missing": miss, "pct": miss / len(raw)}))
for c, n in miss.items():
    log(f"Missing: {c}", f"{n} ({n/len(raw):.0%})",
        {"Minimum Kms To Be Covered In A Day": "Constant 250 km/day policy → used as SLA benchmark",
         "Driver Mobile No": "Dropped (PII, not needed)", "Driver Name": "Dropped (PII, not needed)",
         "Vehicle Type": "Map to 'Unknown' class", "Transportation Distance (KM)": "Impute = straight-line × median road factor (flagged)",
         "Actual ETA": "Excluded from lead-time stats (Ontime flag kept)"}.get(c, "Leave as missing"), int(n))

1899-12-30 placeholder dates: 8 | 'Unknown' carrier: 217 | 'Xxxxxx' material: 14


,missing,pct
Minimum Kms To Be Covered In A Day,2645,0.738
Driver Mobile No,1024,0.286
Vehicle Type,764,0.213
Driver Name,317,0.088
Transportation Distance (KM),148,0.041
Actual ETA,25,0.007
Data Ping time,1,0.000
Curren Location Longitude,1,0.000
Current Location Latitude,1,0.000


In [4]:
# 5. Duplicates
dup_ids = raw[raw["Booking ID"].duplicated(keep=False)].sort_values("Booking ID")
display(dup_ids[["Booking ID", "Gps Provider", "Vehicle Registration", "Customer Name", "Material Shipped", "Ontime"]])
log("Duplicate Booking ID", f"{raw['Booking ID'].duplicated().sum()} extra rows on 2 bookings (same truck & times, different material lines)",
    "Keep first row per booking (trip grain)", int(raw["Booking ID"].duplicated().sum()))

# 6. Inconsistent text (casing / spacing) in locations
for c in ["Origin Location", "Destination Location"]:
    n_raw, n_norm = raw[c].nunique(), raw[c].str.strip().str.title().str.replace(r"\s*,\s*", ", ", regex=True).nunique()
    print(f"{c}: {n_raw} raw spellings → {n_norm} after normalising case & spacing")
    log(f"Inconsistent text: {c}", f"{n_raw} → {n_norm} distinct after normalising", "Title-case, trim, standardise separators", n_raw - n_norm)

,Booking ID,Gps Provider,Vehicle Registration,Customer Name,Material Shipped,Ontime
820,MVCV0000759/082021,Consent Track,KA51A7682,Praveen Engineering Products India Pvt L,Rectifier,No
821,MVCV0000759/082021,Manual,KA51A7682,Praveen Engineering Products India Pvt L,Rectifier Arrangement,No
881,MVCV0000798/082021,Consent Track,HR47C8006,Hi-Tech Gears Ltd,Iut Part,No
882,MVCV0000798/082021,Consent Track,HR47C8006,Rico Auto Industries Ltd,Sol. Relay With Cable Assy,No
883,MVCV0000798/082021,Consent Track,HR47C8006,Talbros Automotive Components Ltd,Solenoid Switch,No


Origin Location: 152 raw spellings → 124 after normalising case & spacing
Destination Location: 353 raw spellings → 353 after normalising case & spacing


In [5]:
# 7. Invalid ranges & business-rule conflicts
lat_ok = lambda s: s.between(6, 37); lon_ok = lambda s: s.between(68, 98)
bad_geo = int((~lat_ok(raw["Origin Location Latitude"]) | ~lon_ok(raw["Origin Location Longitude"]) |
               ~lat_ok(raw["Destination Location Latitude"]) | ~lon_ok(raw["Destination Location Longitude"])).sum())
dist0 = int((raw["Transportation Distance (KM)"] == 0).sum())
b = parsed["Booking Date"]
lead_h = (parsed["Actual ETA"] - b).dt.total_seconds() / 3600
neg_lead, long_lead = int((lead_h < 0).sum()), int((lead_h > 30 * 24).sum())
calc_ontime = np.where(parsed["Actual ETA"] <= parsed["Planned ETA"], "Yes", "No")
conflict = int(((raw["Ontime"] != calc_ontime) & parsed["Actual ETA"].notna()).sum())
print(f"Coordinates outside India box: {bad_geo} | distance = 0: {dist0} | negative lead time: {neg_lead} | lead time > 30 days: {long_lead} | Ontime flag vs ETA conflicts: {conflict}")
log("Coordinates outside India bounding box", f"{bad_geo} rows", "None needed" if bad_geo == 0 else "Flag", bad_geo)
log("Distance = 0 km", f"{dist0} rows", "Impute from coordinates (flagged)", dist0)
log("Negative lead time (Actual ETA < Booking)", f"{neg_lead} rows", "Flag invalid, exclude from lead-time stats", neg_lead)
log("Lead time > 30 days", f"{long_lead} rows (stale GPS / late closure)", "Flag invalid, exclude from lead-time stats", long_lead)
log("Ontime flag vs Actual<=Planned ETA", f"{conflict} conflicts (99.7% agree)", "Keep source flag as the business definition", conflict)
log("In-Full information", "No ordered vs delivered quantity in the data", "OTIF measured as On-Time only (documented assumption)")
log("Freight cost & shipment weight", "Not in the data", "Modelled proxies: rate card by vehicle class; capacity from vehicle type")

dq_df = pd.DataFrame(dq)
dq_df.to_csv(OUTPUTS / "data_quality_log.csv", index=False)
dq_df

Coordinates outside India box: 0 | distance = 0: 18 | negative lead time: 9 | lead time > 30 days: 200 | Ontime flag vs ETA conflicts: 1


,Check,Result,Rows affected,Action
0,Column naming,Typo 'Curren Location Longitude'; names differ...,NaN,Rename to clean PascalCase names
1,Mixed types in date columns,datetime + str + float in the same column,8.000,pd.to_datetime(errors='coerce')
2,Placeholder dates (1899-12-30),Excel zero-date used as 'blank',8.000,Set to NaT
3,Placeholder carrier 'Unknown',Carrier not recorded,217.000,Keep as 'Unknown carrier' (own category)
4,Placeholder material 'Xxxxxx',Material not recorded,14.000,Map to 'Unspecified'
5,Missing: Minimum Kms To Be Covered In A Day,2645 (74%),"2,645.000",Constant 250 km/day policy → used as SLA bench...
6,Missing: Driver Mobile No,1024 (29%),"1,024.000","Dropped (PII, not needed)"
7,Missing: Vehicle Type,764 (21%),764.000,Map to 'Unknown' class
8,Missing: Driver Name,317 (9%),317.000,"Dropped (PII, not needed)"
9,Missing: Transportation Distance (KM),148 (4%),148.000,Impute = straight-line × median road factor (f...


### Data quality verdict
The dataset needs **real cleaning work**:
- **Mixed types and placeholder dates** (Excel's 1899-12-30 "zero date").
- **Free-text locations and materials** (712 distinct material descriptions).
- **Heavy missingness** in vehicle and driver fields.
- **A few duplicated bookings.**
- **Implausible lead times** (stale GPS closures).

Two KPI inputs requested in the brief are **not present at all**:
- **In-Full quantities.** OTIF is therefore measured as on-time only.
- **Freight cost / shipment weight.** These are modelled through transparent, adjustable proxies.

## Part III: Cleaning & Feature Engineering

In [6]:
df = raw.rename(columns={
    "Gps Provider": "GpsProvider", "Booking ID": "BookingID", "Shipment Type": "ShipmentType", "Booking Date": "BookingDate",
    "Vehicle Registration": "VehicleReg", "Origin Location": "Origin", "Destination Location": "Destination",
    "Origin Location Latitude": "OriginLat", "Origin Location Longitude": "OriginLon",
    "Destination Location Latitude": "DestLat", "Destination Location Longitude": "DestLon",
    "Data Ping time": "PingTime", "Planned ETA": "PlannedETA", "Current Location": "CurrentLocation", "Actual ETA": "ActualETA",
    "Current Location Latitude": "CurrentLat", "Curren Location Longitude": "CurrentLon", "Ontime": "Ontime",
    "Trip Start Date": "TripStart", "Trip End Date": "TripEnd", "Transportation Distance (KM)": "DistanceKm",
    "Vehicle Type": "VehicleType", "Minimum Kms To Be Covered In A Day": "MinKmPerDay", "Driver Name": "DriverName",
    "Driver Mobile No": "DriverMobile", "Customer Name": "Customer", "Supplier Name": "Carrier", "Material Shipped": "Material"})

# Dates: coerce mixed types, kill the 1899 placeholder
for c in ["BookingDate", "PingTime", "PlannedETA", "ActualETA", "TripStart", "TripEnd"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
    df.loc[df[c].dt.year < 1950, c] = pd.NaT

# De-duplicate to one row per booking (trip grain) and drop PII
df = df.drop_duplicates("BookingID", keep="first").drop(columns=["DriverMobile", "DriverName"]).reset_index(drop=True)

# Text normalisation
def norm(s):
    return s.astype(str).str.strip().str.replace(r"\s*,\s*", ", ", regex=True).str.replace(r"\s+", " ", regex=True).str.title()
for c in ["Origin", "Destination", "Customer", "Carrier", "Material", "GpsProvider", "VehicleType"]:
    df[c] = norm(df[c]).replace({"Nan": np.nan})
df["Carrier"] = df["Carrier"].replace({"Unknown": "Unknown Carrier"})
df["Customer"] = df["Customer"].str.replace(r"\b(Pvt|Private) L(t|td)?$", "Pvt Ltd", regex=True)
print(df.shape)

(3582, 26)


In [7]:
# Origin / destination city & state parsed from 'Place, City, State'
STATE_FIX = {"Chattisgarh": "Chhattisgarh", "Pondicherry": "Puducherry", "Jammu & Kashmir": "Jammu And Kashmir",
             "Dadra & Nagar Haveli": "Dadra And Nagar Haveli", "Central Development Region": "Nepal (Central)"}
ZONE = {"North": ["Delhi", "Haryana", "Punjab", "Himachal Pradesh", "Jammu And Kashmir", "Uttarakhand", "Chandigarh", "Rajasthan", "Uttar Pradesh"],
        "West": ["Maharashtra", "Gujarat", "Goa", "Dadra And Nagar Haveli"],
        "South": ["Tamil Nadu", "Karnataka", "Kerala", "Telangana", "Andhra Pradesh", "Puducherry"],
        "East": ["West Bengal", "Odisha", "Bihar", "Jharkhand"],
        "Central": ["Madhya Pradesh", "Chhattisgarh"],
        "North-East": ["Assam", "Meghalaya"], "International": ["Nepal (Central)"]}
STATE2ZONE = {s: z for z, ss in ZONE.items() for s in ss}
for side in ["Origin", "Destination"]:
    parts = df[side].str.split(", ")
    df[f"{side}State"] = parts.str[-1].replace(STATE_FIX)
    df[f"{side}City"] = parts.apply(lambda p: p[-2] if isinstance(p, list) and len(p) >= 2 else np.nan)
    df[f"{side}Zone"] = df[f"{side}State"].map(STATE2ZONE).fillna("Other")
df["Route"] = df.OriginState + " → " + df.DestinationState
df["Lane"] = df.OriginZone + " → " + df.DestinationZone
print(df.DestinationZone.value_counts())

DestinationZone
South            1757
North             711
West              607
East              395
North-East        105
Central             6
International       1
Name: count, dtype: int64


In [8]:
# Material type: 712 free-text descriptions → 6 categories (keyword rules, checked by hand)
MATERIAL_RULES = [
    ("Returnable Packaging (Empty)", r"\bempty\b|\btray|\bbin\b|\bfoam\b|\blids?\b|\bpallet"),
    ("Electrical Components", r"solenoid|rectifier|alternator|starter|switch|relay|regulator|wiper|motor|brush|thermal|harness|lamp|wiring|armature|coil|sensor|fuse|cable"),
    ("Mechanical / Drivetrain", r"bearing|spring|shaft|bolt|gear|spindle|bush|screw|washer|pinion|ring|sleeve|yoke|pole|drive|clutch|axle|brake|link|plate|bracket|bkt|pin\b|nut\b|valve|pulley|housing|cross member|synchron|velocity"),
    ("General Auto Parts", r"auto parts?|iut parts?|spare"),
    ("Unspecified", r"^x+$|^nan$"),
]
def material_category(m):
    s = str(m).lower()
    for cat, pat in MATERIAL_RULES:
        if re.search(pat, s):
            return cat
    return "Other Industrial"
df["MaterialCategory"] = df.Material.map(material_category)
df["IsEmptyReturn"] = (df.MaterialCategory == "Returnable Packaging (Empty)").astype(int)
display(df.MaterialCategory.value_counts())
display(df.groupby("MaterialCategory").Material.agg(lambda s: ", ".join(s.value_counts().index[:5])).to_frame("Top descriptions"))

MaterialCategory
General Auto Parts              1234
Electrical Components            767
Mechanical / Drivetrain          607
Other Industrial                 582
Returnable Packaging (Empty)     378
Unspecified                       14
Name: count, dtype: int64

,Top descriptions
MaterialCategory,
Electrical Components,"Grs Starter, Solenoid Assembly, Wiper Motor Li..."
General Auto Parts,"Auto Parts, Iut Parts, Iut Part"
Mechanical / Drivetrain,"Ball Bearing, Spindle Assy, Yoke & Pole Assy, ..."
Other Industrial,"Plug Modg, Lucar Plade, Arrangement-26071777, ..."
Returnable Packaging (Empty),"Empty Trays, Empty Bin, Empty Foam, Empty Cb, ..."
Unspecified,Xxxxxx


In [9]:
# Vehicle: capacity (tonnes) parsed from the free-text type, then a vehicle class
df["CapacityMT"] = df.VehicleType.str.extract(r"(\d+(?:\.\d+)?)\s*Mt\b", expand=False, flags=re.I).astype(float)
def vclass(row):
    vt, cap = str(row.VehicleType).lower(), row.CapacityMT
    if vt == "nan": return "Unknown"
    if "container" in vt: return "Container"
    if "trailer" in vt or (cap and cap >= 25): return "Trailer (25-35 MT)"
    if cap and cap >= 10: return "HCV (10-24 MT)"
    if cap and cap >= 5: return "MCV (5-9 MT)"
    if cap and cap > 0: return "LCV (< 5 MT)"
    return "HCV (10-24 MT)" if "32 ft" in vt else "Other"
df["VehicleClass"] = df.apply(vclass, axis=1)
print(df.VehicleClass.value_counts(dropna=False))

VehicleClass
HCV (10-24 MT)        1053
Trailer (25-35 MT)     944
Unknown                761
MCV (5-9 MT)           626
Container              123
LCV (< 5 MT)            75
Name: count, dtype: int64


In [10]:
# Distance: GPS-route km vs. straight-line (haversine) km
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))
df["CrowKm"] = haversine(df.OriginLat, df.OriginLon, df.DestLat, df.DestLon).round(1)
valid = (df.DistanceKm > 0) & (df.CrowKm > 5)
ROAD_FACTOR = float((df.loc[valid, "DistanceKm"] / df.loc[valid, "CrowKm"]).median())
df["DistanceImputed"] = (df.DistanceKm.isna() | (df.DistanceKm == 0)).astype(int)
df.loc[df.DistanceImputed == 1, "DistanceKm"] = (df.CrowKm * ROAD_FACTOR).round(0)
df["RouteToCrowRatio"] = np.where(df.CrowKm > 5, (df.DistanceKm / df.CrowKm).round(2), np.nan)
print(f"Median road factor (route km / straight-line km): {ROAD_FACTOR:.2f}  | distances imputed: {df.DistanceImputed.sum()}")

df["DistanceBand"] = pd.cut(df.DistanceKm, [-1, 100, 300, 700, 1200, 2000, 5000],
                            labels=["< 100 km", "100-300 km", "300-700 km", "700-1,200 km", "1,200-2,000 km", "> 2,000 km"])

Median road factor (route km / straight-line km): 1.31  | distances imputed: 166


In [11]:
# Time features
H = lambda a, b: (a - b).dt.total_seconds() / 3600
df["IsOnTime"] = (df.Ontime == "Yes").astype(int)
df["PlannedTransitHrs"] = H(df.PlannedETA, df.BookingDate).round(1)
df["LeadTimeHrs"] = H(df.ActualETA, df.BookingDate).round(1)
df["DelayHrs"] = H(df.ActualETA, df.PlannedETA).round(1)
df["ValidLeadTime"] = (df.LeadTimeHrs.between(0, 30 * 24)).astype(int)
df.loc[df.ValidLeadTime == 0, ["LeadTimeHrs", "DelayHrs"]] = np.nan
df["LeadTimeDays"] = (df.LeadTimeHrs / 24).round(2)

# SLA feasibility: the dataset's own policy is a minimum of 250 km per day (= 10.4 km/h average incl. stops)
KM_PER_DAY = 250
df["PlannedSpeedKmh"] = np.where(df.PlannedTransitHrs > 0, (df.DistanceKm / df.PlannedTransitHrs).round(1), np.nan)
df["RealisticTransitHrs"] = (df.DistanceKm / KM_PER_DAY * 24).round(1)
df["SLAFeasible"] = (df.PlannedTransitHrs >= df.RealisticTransitHrs).astype(int)

df["BookingDay"] = df.BookingDate.dt.normalize()
df["BookingMonth"] = df.BookingDate.dt.to_period("M").dt.to_timestamp()
df["BookingWeekday"] = df.BookingDate.dt.day_name()
df["IsWeekend"] = df.BookingDate.dt.dayofweek.isin([5, 6]).astype(int)

df[["BookingID", "DistanceKm", "PlannedTransitHrs", "RealisticTransitHrs", "PlannedSpeedKmh", "SLAFeasible", "LeadTimeHrs", "DelayHrs", "IsOnTime"]].head()

,BookingID,DistanceKm,PlannedTransitHrs,RealisticTransitHrs,PlannedSpeedKmh,SLAFeasible,LeadTimeHrs,DelayHrs,IsOnTime
0,AEIBK2027469,"1,290.000",100.000,123.800,12.900,0,48.700,-51.300,1
1,VCV00014153/082021,29.000,100.000,2.800,0.300,1,20.900,-79.100,1
2,VCV00014063/082021,21.000,100.000,2.000,0.200,1,21.200,-78.800,1
3,VCV00014741/082021,20.000,100.000,1.900,0.200,1,11.000,-89.000,1
4,AEIBK2027446,"1,900.000",100.000,182.400,19.000,0,69.200,-30.800,1


In [12]:
# Modelled freight cost (the data has no cost field) - an adjustable, clearly-labelled assumption
RATE_CARD_INR_PER_KM = {"LCV (< 5 MT)": 22, "MCV (5-9 MT)": 32, "HCV (10-24 MT)": 42, "Container": 48,
                        "Trailer (25-35 MT)": 58, "Other": 42, "Unknown": 42}
DEFAULT_CAP = {"LCV (< 5 MT)": 2, "MCV (5-9 MT)": 7, "HCV (10-24 MT)": 14, "Container": 9, "Trailer (25-35 MT)": 30, "Other": 14, "Unknown": 14}
df["RateInrPerKm"] = df.VehicleClass.map(RATE_CARD_INR_PER_KM)
df["CapacityUsedMT"] = df.CapacityMT.fillna(df.VehicleClass.map(DEFAULT_CAP))
df["EstFreightCostINR"] = (df.DistanceKm * df.RateInrPerKm).round(0)
df["EstCostPerTonneINR"] = (df.EstFreightCostINR / df.CapacityUsedMT).round(0)
df["EstCostPerTonneKmINR"] = (df.RateInrPerKm / df.CapacityUsedMT).round(2)
pd.Series(RATE_CARD_INR_PER_KM, name="Assumed rate (INR/km)").to_frame()

,Assumed rate (INR/km)
LCV (< 5 MT),22
MCV (5-9 MT),32
HCV (10-24 MT),42
Container,48
Trailer (25-35 MT),58
Other,42
Unknown,42


## Part IV: Network KPI Overview

In [13]:
period = df[(df.BookingDate >= "2019-12-01") & (df.BookingDate < "2020-09-01")]
kpi = pd.Series({
    "Trips": len(df), "Trips in core period (Dec-19 to Aug-20)": len(period),
    "OTIF % (on-time)": df.IsOnTime.mean(),
    "Median lead time (days)": df.LeadTimeDays.median(),
    "Avg lead time (days)": df.LeadTimeDays.mean(),
    "Median delay of late trips (hrs)": df.loc[df.IsOnTime == 0, "DelayHrs"].median(),
    "Total distance (km)": df.DistanceKm.sum(),
    "Median distance (km)": df.DistanceKm.median(),
    "Carriers": df.Carrier.nunique(), "Customers": df.Customer.nunique(),
    "Trips with infeasible SLA": 1 - df.SLAFeasible.mean(),
    "Empty-return trips": df.IsEmptyReturn.mean(),
    "Est. freight spend (INR M)": df.EstFreightCostINR.sum() / 1e6,
})
kpi.to_frame("Value")

,Value
Trips,"3,582.000"
Trips in core period (Dec-19 to Aug-20),"3,571.000"
OTIF % (on-time),0.386
Median lead time (days),4.285
Avg lead time (days),5.474
Median delay of late trips (hrs),99.400
Total distance (km),"3,095,065.800"
Median distance (km),525.000
Carriers,192.000
Customers,34.000


## Part V: Q1, OTIF Health & Trend

In [14]:
m = (period.groupby("BookingMonth").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"),
                                         MedianKm=("DistanceKm", "median"), Feasible=("SLAFeasible", "mean"),
                                         TopCustomer=("Customer", lambda s: s.value_counts().index[0])).reset_index())
m.to_csv(OUTPUTS / "monthly_otif.csv", index=False)
fig = go.Figure()
fig.add_bar(x=m.BookingMonth, y=m.Trips, name="Trips", marker_color="#CBD5E1", yaxis="y2")
fig.add_scatter(x=m.BookingMonth, y=m.OTIF, name="OTIF %", mode="lines+markers+text", text=[f"{v:.0%}" for v in m.OTIF],
                textposition="top center", line=dict(color=NAVY, width=3))
fig.update_layout(title="Monthly OTIF % (on-time) and trip volume", yaxis=dict(tickformat=".0%", range=[0, 1]),
                  yaxis2=dict(overlaying="y", side="right", showgrid=False, title="Trips"), legend=dict(orientation="h", y=-0.15))
fig.show()
m

,BookingMonth,Trips,OTIF,MedianKm,Feasible,TopCustomer
0,2019-12-01,179,0.000,69.500,0.855,Larsen & Toubro Limited
1,2020-01-01,358,0.011,90.000,0.911,Larsen & Toubro Limited
2,2020-02-01,68,0.000,75.750,0.824,Larsen & Toubro Limited
3,2020-03-01,44,0.000,65.000,0.955,Larsen & Toubro Limited
4,2020-05-01,55,0.000,109.000,0.873,Larsen & Toubro Limited
5,2020-06-01,496,0.234,"1,290.000",0.284,Ford India Private Limited
6,2020-07-01,942,0.312,"1,290.000",0.285,Ford India Private Limited
7,2020-08-01,1429,0.672,310.000,0.685,Daimler India Commercial Vehicles Pvt Ltd


> **INSIGHT Q1: network OTIF is only 38.6%. The trend looks better than it is.**
> - **38.6% of 3,582 trips arrived on time**, far below any 3PL benchmark (≥ 90%).
> - The monthly trend splits into two regimes:
>   - **Dec-2019 to May-2020: 0–1% on time.** The book is almost only Larsen & Toubro short site shuttles (median 70–110 km).
>   - **Jun to Aug 2020: OTIF climbs from 23% to 31% to 67%.**
> - The August jump is **mostly mix**: Daimler / Ashok Leyland short-haul lanes in Tamil Nadu (73–91% on time) took over the volume. The long-haul lanes did not improve. **The trend is real only for short-haul.**
> - The L&T period is a separate story. **85–95% of those plans were feasible on distance**, yet almost none arrived on time. The planned window was a few hours, but trucks took days. That is **dwell / loading time that is not in the plan**, not driving time (see Q2).

## Part VI: Q2, Route Bottlenecks: Where Is Lead Time Abnormal, and Why?

In [15]:
zone = (df.groupby("DestinationZone").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"),
          MedianLeadDays=("LeadTimeDays", "median"), MedianKm=("DistanceKm", "median"),
          MedianDelayHrs=("DelayHrs", "median"), Feasible=("SLAFeasible", "mean"))
          .query("Trips >= 10").sort_values("MedianLeadDays", ascending=False))
display(zone)
routes = (df.groupby("Route").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"), MedianLeadDays=("LeadTimeDays", "median"),
            MedianKm=("DistanceKm", "median"), MedianPlannedHrs=("PlannedTransitHrs", "median"), Feasible=("SLAFeasible", "mean"))
            .query("Trips >= 25").sort_values("MedianLeadDays", ascending=False))
routes.to_csv(OUTPUTS / "route_bottlenecks.csv")
routes.head(12)

,Trips,OTIF,MedianLeadDays,MedianKm,MedianDelayHrs,Feasible
DestinationZone,,,,,,
North-East,105,0.219,9.430,307.000,127.700,0.657
North,711,0.118,6.910,"1,775.000",104.100,0.314
West,607,0.367,4.820,"1,170.000",27.900,0.407
East,395,0.286,4.770,210.000,43.400,0.894
South,1757,0.533,2.855,310.000,-4.800,0.639


,Trips,OTIF,MedianLeadDays,MedianKm,MedianPlannedHrs,Feasible
Route,,,,,,
Maharashtra → Assam,26,0.000,12.460,"2,750.000",100.000,0.000
Tamil Nadu → Himachal Pradesh,28,0.036,9.280,"2,600.000",100.000,0.000
Maharashtra → Rajasthan,73,0.014,7.690,"1,178.000",21.400,0.000
Maharashtra → Andhra Pradesh,28,0.000,7.580,900.000,100.000,1.000
Karnataka → Haryana,36,0.194,7.365,"2,211.000",35.750,0.000
Tamil Nadu → Haryana,195,0.041,7.160,"2,300.000",100.000,0.000
Uttar Pradesh → Uttar Pradesh,84,0.083,6.860,68.500,15.400,0.964
Odisha → Odisha,102,0.137,6.250,30.000,23.200,0.941
Tamil Nadu → Gujarat,128,0.094,6.075,"1,900.000",100.000,0.000


In [16]:
band = (df.groupby("DistanceBand").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"), MedianLeadDays=("LeadTimeDays", "median"),
          MedianPlannedHrs=("PlannedTransitHrs", "median"), RealisticHrs=("RealisticTransitHrs", "median"), Feasible=("SLAFeasible", "mean")))
display(band)
feas = df.groupby("SLAFeasible").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"))
display(feas)
fig = px.scatter(df.dropna(subset=["PlannedTransitHrs"]), x="DistanceKm", y="PlannedTransitHrs", color=df.Ontime.loc[df.PlannedTransitHrs.notna()],
                 color_discrete_map={"Yes": GREEN, "No": RED}, opacity=0.55,
                 title="Planned transit time vs. distance: the SLA ignores distance (dashed = 250 km/day policy)",
                 labels={"color": "On time", "DistanceKm": "Distance (km)", "PlannedTransitHrs": "Planned transit (hrs)"})
xs = np.linspace(0, 3000, 50)
fig.add_scatter(x=xs, y=xs / KM_PER_DAY * 24, mode="lines", line=dict(dash="dash", color=NAVY), name="Realistic (250 km/day)")
fig.update_yaxes(range=[0, 400]); fig.show()

,Trips,OTIF,MedianLeadDays,MedianPlannedHrs,RealisticHrs,Feasible
DistanceBand,,,,,,
< 100 km,819,0.385,2.850,7.600,2.900,0.927
100-300 km,714,0.678,0.930,100.000,17.000,0.961
300-700 km,427,0.541,3.005,100.000,38.400,0.808
"700-1,200 km",522,0.335,5.295,100.000,101.300,0.433
"1,200-2,000 km",564,0.193,5.960,100.000,152.600,0.000
"> 2,000 km",536,0.125,6.620,100.000,232.800,0.000


,Trips,OTIF
SLAFeasible,,
0,1566,0.167
1,2016,0.556


> **INSIGHT Q2: two different bottlenecks, with two different causes.**
>
> **(a) Long-haul: the SLA ignores distance.**
> - Most bookings get a **flat ~100-hour ETA** whatever the distance.
> - The dataset's own policy is 250 km/day, so a 2,300 km trip needs about 221 hours.
> - **No trip over 1,200 km has a feasible plan**, so OTIF there collapses to **12–19%**.
> - Worst lanes: **Tamil Nadu → Haryana** (195 trips, 4% OTIF, 7.2-day median lead time) and **Maharashtra → Assam** (12.5 days, 0%).
> - SLA feasibility is the single strongest driver: **55.6% OTIF when the plan is feasible vs 16.7% when it is not** (Cramér's V = 0.40).
>
> **(b) Short-haul: dwell time.**
> - **Uttar Pradesh → Uttar Pradesh** (68 km): 6.9-day median lead time. **Odisha → Odisha** (30 km): 6.3 days.
> - Trucks are waiting at plants and sites, not driving.
> - The **North-East** (9.4 days for a median 307 km) adds terrain and permit delays on top.
>
> **By destination zone:** North has the longest distances and the worst OTIF (11.8%, 6.9 days). South is the healthy core (53%, 2.9 days).

## Part VII: Q3, Carrier (Supplier) Performance

In [17]:
# Raw on-time % is unfair: carriers on long-haul lanes look worse. Benchmark each carrier against the
# network on-time rate *for the same distance band and SLA feasibility* (expected), then compare.
cell = df.groupby(["DistanceBand", "SLAFeasible"]).IsOnTime.mean().rename("ExpectedOnTime")
df = df.merge(cell, left_on=["DistanceBand", "SLAFeasible"], right_index=True, how="left")
car = (df.groupby("Carrier").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"), Expected=("ExpectedOnTime", "mean"),
          MedianKm=("DistanceKm", "median"), MedianDelayHrs=("DelayHrs", "median"), Customers=("Customer", "nunique"))
         .query("Trips >= 30"))
car["VsExpected_pp"] = (car.OTIF - car.Expected) * 100
car["Verdict"] = np.select([car.index == "Unknown Carrier", car.VsExpected_pp <= -15, car.VsExpected_pp >= 15],
                           ["Data gap - fix carrier capture", "Exit / re-tender", "Preferred - grow"], "Monitor")
car = car.sort_values("VsExpected_pp")
car.to_csv(OUTPUTS / "carrier_scorecard.csv")
car

,Trips,OTIF,Expected,MedianKm,MedianDelayHrs,Customers,VsExpected_pp,Verdict
Carrier,,,,,,,,
K M R Transport,31,0.000,0.434,65.000,101.100,1,-43.426,Exit / re-tender
A P R Trailler Service,78,0.103,0.513,66.000,126.600,1,-41.014,Exit / re-tender
K.Ramachandran Transports,61,0.049,0.456,90.000,107.300,1,-40.672,Exit / re-tender
Shri Sai Enterprises,32,0.062,0.438,65.000,113.350,1,-37.544,Exit / re-tender
Baba Lingaraj Enterprises Pvt Ltd,78,0.179,0.512,81.500,72.000,1,-33.257,Exit / re-tender
Sunita Carriers Private Limited,160,0.169,0.494,210.000,75.600,1,-32.496,Exit / re-tender
S.B.Transport Company,38,0.000,0.294,25.000,104.800,1,-29.391,Exit / re-tender
Shri Ganesh Traders And Transporter,75,0.347,0.634,161.000,32.700,1,-28.759,Exit / re-tender
Rajdhani Roadways,64,0.000,0.245,"1,267.000",105.000,1,-24.499,Exit / re-tender


In [18]:
fig = px.bar(car.reset_index(), x="VsExpected_pp", y="Carrier", orientation="h", color="Verdict",
             color_discrete_map={"Exit / re-tender": RED, "Monitor": GREY, "Preferred - grow": GREEN, "Data gap - fix carrier capture": ORANGE},
             title="Carrier on-time vs. expected for the same distance & SLA mix (pp, carriers with ≥ 30 trips)")
fig.update_layout(yaxis=dict(categoryorder="total ascending"), height=650); fig.show()

> **INSIGHT Q3: rank carriers against their lane, not in absolute terms.**
> - Raw OTIF punishes carriers that run long-haul lanes. *Ekta Transport* (238 trips, 21.8%) and *Trans Cargo India* (13.1%) look terrible, but they perform **exactly as expected for their 1,900–2,400 km lanes** (−3 and −1 pp). **Fix their SLAs, don't fire them.**
> - **Exit / re-tender (≥ 15 pp below expected, ≥ 30 trips):** 12 carriers, about 800 trips (22% of volume). The worst:
>   - *K M R Transport*: 0% on time, −43 pp
>   - *A P R Trailler Service*: 10%, −41 pp
>   - *K. Ramachandran Transports*: 5%, −41 pp
>   - *Sunita Carriers*: 17% on 160 trips, −32 pp
>   - *Rajdhani Roadways*: 0% on 64 trips
> - **Preferred, grow volume:** the best performers by points above expected:
>   - *Namakkal Sri Anjinaya*: 100%, +59 pp
>   - *Eswar Transport*: 95%, +50 pp
>   - *Arvinth Transport*: 76%, +40 pp
>   - *Sr Transports*: 100% on 113 trips
>   - *VJ Logistics*: 62% on 174 trips, +26 pp
> - **217 trips have no carrier recorded** ("Unknown", 11.5% on time). That is a data-governance gap: you cannot manage a vendor you cannot see.

## Part VIII: Q4, Is Delay Linked to Material Type?

In [19]:
mat = (df.groupby("MaterialCategory").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"), MedianKm=("DistanceKm", "median"),
         Feasible=("SLAFeasible", "mean"), Expected=("ExpectedOnTime", "mean")).sort_values("OTIF"))
mat["VsExpected_pp"] = (mat.OTIF - mat.Expected) * 100
display(mat)
ct = pd.crosstab(df.MaterialCategory, df.IsOnTime)
chi2, p, dof, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
ct2 = pd.crosstab(df.SLAFeasible, df.IsOnTime); chi2b, pb, _, _ = stats.chi2_contingency(ct2)
v2 = np.sqrt(chi2b / ct2.values.sum())
print(f"Material vs on-time: chi2={chi2:.1f}, p={p:.2e}, Cramér's V={cramers_v:.2f}")
print(f"SLA feasibility vs on-time: chi2={chi2b:.1f}, p={pb:.2e}, Cramér's V={v2:.2f}")

,Trips,OTIF,MedianKm,Feasible,Expected,VsExpected_pp
MaterialCategory,,,,,,
Unspecified,14,0.071,"1,470.000",0.429,0.329,-25.756
General Auto Parts,1234,0.328,601.000,0.549,0.376,-4.801
Electrical Components,767,0.381,451.000,0.549,0.380,0.082
Returnable Packaging (Empty),378,0.405,585.000,0.558,0.387,1.815
Other Industrial,582,0.411,386.000,0.577,0.387,2.332
Mechanical / Drivetrain,607,0.479,370.000,0.601,0.411,6.884


Material vs on-time: chi2=47.7, p=4.00e-09, Cramér's V=0.12
SLA feasibility vs on-time: chi2=561.0, p=5.05e-124, Cramér's V=0.40


> **INSIGHT Q4: material type is a weak signal; the lane and the SLA are the real drivers.**
> - On-time rate does differ by material (χ² p < 0.001). *General Auto Parts* is lowest at 32.8% and *Mechanical / Drivetrain* is highest at 47.9%.
> - But the effect size is small: **Cramér's V = 0.12, vs 0.40 for SLA feasibility**.
> - After controlling for distance and SLA, material categories sit within about ±7 pp of expected.
> - Delays follow **where and how far** the cargo goes, **not what it is**. Material-based routing would not move OTIF.

## Part IX: Q5, Load Utilisation (Proxy)

In [20]:
# No weight / volume field exists, so we use two observable proxies:
# (a) trips carrying only returnable empty packaging (trays, bins, foam) = running 'empty' for the customer
# (b) heavy trailers used on very short shuttles (capacity likely under-used)
util = pd.Series({
    "Empty-return trips (share)": df.IsEmptyReturn.mean(),
    "Empty-return trips (count)": df.IsEmptyReturn.sum(),
    "Trailer (25-35 MT) trips < 100 km (share of trailer trips)": ((df.VehicleClass == "Trailer (25-35 MT)") & (df.DistanceKm < 100)).sum() / (df.VehicleClass == "Trailer (25-35 MT)").sum(),
    "Est. spend on empty-return trips (INR M)": df.loc[df.IsEmptyReturn == 1, "EstFreightCostINR"].sum() / 1e6,
})
display(util.to_frame("Value"))
vc = df.groupby("VehicleClass").agg(Trips=("BookingID", "count"), MedianKm=("DistanceKm", "median"), EmptyShare=("IsEmptyReturn", "mean"),
                                    OTIF=("IsOnTime", "mean"), TopCustomer=("Customer", lambda s: s.value_counts().index[0]))
vc

,Value
Empty-return trips (share),0.106
Empty-return trips (count),378.000
Trailer (25-35 MT) trips < 100 km (share of trailer trips),0.550
Est. spend on empty-return trips (INR M),13.373


,Trips,MedianKm,EmptyShare,OTIF,TopCustomer
VehicleClass,,,,,
Container,123,325.000,0.106,0.228,Ericsson India Private Limited
HCV (10-24 MT),1053,"1,851.000",0.110,0.235,Ford India Private Limited
LCV (< 5 MT),75,400.000,0.120,0.773,Ashok Leyland Limited
MCV (5-9 MT),626,"1,279.500",0.109,0.173,Ford India Private Limited
Trailer (25-35 MT),944,88.800,0.085,0.210,Larsen & Toubro Limited
Unknown,761,110.000,0.121,0.975,Daimler India Commercial Vehicles Pvt Ltd


> **INSIGHT Q5: load utilisation (proxy). About 1 in 10 trips runs empty for the customer.**
> - The data has no weight field, so two observable proxies are used:
>   - **10.6% of trips (378) carry only returnable empty packaging** (trays, bins, foam). That is about **₹13.4 M of modelled spend** on reverse logistics.
>   - **55% of the 25–35 MT trailer trips are shuttles under 100 km**, mostly L&T sites. A 35-tonne trailer on a 70 km shuttle is very likely under-loaded.
> - **Actions:**
>   - Pair empty-packaging returns with forward loads (backhaul matching).
>   - Right-size short shuttles to MCV / HCV.
>   - Start capturing **actual loaded weight** at dispatch so true utilisation can be measured.

## Part X: Q6, GPS Distance Anomalies

In [21]:
df["AnomalyType"] = np.select(
    [(df.CrowKm <= 5) & (df.DistanceKm > 50),
     df.RouteToCrowRatio < 0.9,
     df.RouteToCrowRatio > 2.5],
    ["Round trip (origin = destination)", "Route shorter than straight line (impossible)", "Route > 2.5x straight line (detour)"],
    default="Normal")
anom = df.AnomalyType.value_counts().to_frame("Trips"); anom["Share"] = anom.Trips / len(df)
display(anom)
display(df[df.AnomalyType != "Normal"].groupby("AnomalyType").IsOnTime.mean().to_frame("OTIF"))
fig = px.scatter(df[df.CrowKm > 5], x="CrowKm", y="DistanceKm", color="AnomalyType", opacity=0.6,
                 color_discrete_map={"Normal": GREY, "Route shorter than straight line (impossible)": RED, "Route > 2.5x straight line (detour)": ORANGE},
                 title=f"GPS route distance vs. straight-line distance (median road factor {ROAD_FACTOR:.2f})")
fig.add_scatter(x=[0, 3000], y=[0, 3000 * ROAD_FACTOR], mode="lines", line=dict(dash="dash", color=NAVY), name="Median road factor")
fig.show()
df.loc[df.AnomalyType != "Normal", ["BookingID", "Origin", "Destination", "CrowKm", "DistanceKm", "RouteToCrowRatio", "Carrier", "AnomalyType"]].head(10)

,Trips,Share
AnomalyType,,
Normal,2724,0.760
Route shorter than straight line (impossible),418,0.117
Round trip (origin = destination),235,0.066
Route > 2.5x straight line (detour),205,0.057


,OTIF
AnomalyType,
Round trip (origin = destination),0.979
Route > 2.5x straight line (detour),0.746
Route shorter than straight line (impossible),0.158


,BookingID,Origin,Destination,CrowKm,DistanceKm,RouteToCrowRatio,Carrier,AnomalyType
6,VCV00014935/082021,"Heromotocorp Ltd, Alwar, Rajasthan","Heromotocorp Ltd, Alwar, Rajasthan",0.000,242.000,NaN,Virs Tempo Service,Round trip (origin = destination)
15,VCV00014096/082021,"Daimler India Commercial Vehicles, Kanchipuram...","Daimler India Commercial Vehicles, Kanchipuram...",0.000,69.000,NaN,Archana Transport,Round trip (origin = destination)
24,VCV00013531/082021,"Daimler India Commercial Vehicles, Kanchipuram...","Daimler India Commercial Vehicles, Kanchipuram...",0.000,69.000,NaN,Susee Transporter,Round trip (origin = destination)
25,VCV00012819/082021,"Daimler India Commercial Vehicles, Kanchipuram...","Daimler India Commercial Vehicles, Kanchipuram...",0.000,931.000,NaN,Eswar Transport,Round trip (origin = destination)
26,VCV00013919/082021,"Daimler India Commercial Vehicles, Kanchipuram...","Daimler India Commercial Vehicles, Kanchipuram...",0.000,70.000,NaN,Eswar Transport,Round trip (origin = destination)
29,VCV00013912/082021,"Daimler India Commercial Vehicles, Kanchipuram...","Daimler India Commercial Vehicles, Kanchipuram...",0.000,61.000,NaN,Arvinth Transport,Round trip (origin = destination)
34,MVCV0001702/082021,"Tvslsl Hub (Chakan), Pune, Maharashtra","Tvslsl Hub (Chakan), Pune, Maharashtra",0.000,353.000,NaN,Tanaji Raosaheb Kolekar,Round trip (origin = destination)
35,VCV00013407/082021,"Daimler India Commercial Vehicles, Kanchipuram...","Daimler India Commercial Vehicles, Kanchipuram...",0.000,103.000,NaN,Arvinth Transport,Round trip (origin = destination)
37,MVCV0001701/082021,"Tvslsl Hub (Chakan), Pune, Maharashtra","Tvslsl Hub (Chakan), Pune, Maharashtra",0.000,353.000,NaN,Tanaji Raosaheb Kolekar,Round trip (origin = destination)
42,MVCV0001651/082021,"Tvslsl Hub (Chakan), Pune, Maharashtra","Tvslsl Hub (Chakan), Pune, Maharashtra",498.100,"1,340.000",2.690,Oms Logistics Pvt Ltd,Route > 2.5x straight line (detour)


> **INSIGHT Q6: one in four trips has a GPS / master-data anomaly.**
> - **76% of trips are normal.** The route is on average 1.31× the straight line, which is typical for Indian roads.
> - **418 trips (11.7%) report a route shorter than the straight line.** That is physically impossible, so the distance or the coordinates are wrong.
>   - These trips are only **15.8% on time**. Corrupted distances feed corrupted ETAs.
> - **205 trips (5.7%) are more than 2.5× the straight line.** These are detours or wrong destination pins and are candidates for route compliance checks.
> - **235 trips (6.6%) start and end at the same point.** They are round trips / milk runs (98% on time), not errors, but they need their own SLA logic.

## Part XI: Q7, Weekday vs. Weekend

In [22]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
wd = (df.groupby("BookingWeekday").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"), MedianKm=("DistanceKm", "median"),
        LongHaulShare=("DistanceKm", lambda s: (s > 1200).mean()), Feasible=("SLAFeasible", "mean"), Expected=("ExpectedOnTime", "mean"))
        .reindex(order))
wd["VsExpected_pp"] = (wd.OTIF - wd.Expected) * 100
display(wd)
fig = px.bar(wd.reset_index(), x="BookingWeekday", y="OTIF", text=wd.OTIF.map("{:.0%}".format), color="LongHaulShare",
             color_continuous_scale="Oranges", title="OTIF by booking weekday (colour = share of long-haul > 1,200 km)")
fig.update_layout(yaxis_tickformat=".0%"); fig.show()

,Trips,OTIF,MedianKm,LongHaulShare,Feasible,Expected,VsExpected_pp
BookingWeekday,,,,,,,
Monday,663,0.460,282.000,0.237,0.671,0.426,3.382
Tuesday,677,0.383,530.000,0.276,0.572,0.390,-0.779
Wednesday,623,0.374,400.000,0.334,0.552,0.377,-0.283
Thursday,650,0.383,481.000,0.318,0.569,0.390,-0.674
Friday,563,0.391,601.000,0.346,0.515,0.372,1.834
Saturday,342,0.292,"1,082.500",0.386,0.424,0.323,-3.016
Sunday,64,0.234,900.000,0.219,0.547,0.407,-17.215


> **INSIGHT Q7: "Wednesday is always late" is a myth. The weekday gap is a mix effect.**
> - Wednesday's OTIF (37.4%) is **in line with its lane mix** (−0.3 pp vs expected).
> - Monday is best (46.0%) because it carries short-haul shuttles (median 282 km).
> - **The real weak days are Saturday** (29.2%, median 1,083 km, 39% long-haul) **and Sunday** (23.4%, −17 pp vs expected on 64 trips). Long-haul loads go out at the weekend into thin weekend operations.
> - **Action:** don't release long-haul loads on Saturday/Sunday without a feasible ETA and weekend dispatch cover.

## Part XII: Q8, Cost Efficiency by Region (Modelled)

In [23]:
cost = (df.groupby("DestinationZone").agg(Trips=("BookingID", "count"), SpendINR_M=("EstFreightCostINR", lambda s: s.sum() / 1e6),
          CostPerTrip=("EstFreightCostINR", "mean"), CostPerTonne=("EstCostPerTonneINR", "median"),
          CostPerTonneKm=("EstCostPerTonneKmINR", "median"), MedianKm=("DistanceKm", "median"), OTIF=("IsOnTime", "mean"))
          .query("Trips >= 10").sort_values("CostPerTonne", ascending=False))
display(cost)
fig = px.scatter(cost.reset_index(), x="MedianKm", y="CostPerTonne", size="Trips", color="OTIF", text="DestinationZone",
                 color_continuous_scale="RdYlGn", title="Modelled freight cost per tonne vs. distance by destination zone")
fig.update_traces(textposition="top center"); fig.show()

,Trips,SpendINR_M,CostPerTrip,CostPerTonne,CostPerTonneKm,MedianKm,OTIF
DestinationZone,,,,,,,
North,711,41.478,"58,338.062","5,628.000",3.000,"1,775.000",0.118
West,607,23.898,"39,371.379","3,597.000",3.000,"1,170.000",0.367
South,1757,48.452,"27,576.399","1,095.000",3.000,310.000,0.533
North-East,105,3.641,"34,678.886",607.000,2.150,307.000,0.219
East,395,5.278,"13,363.203",368.000,2.150,210.000,0.286


> **INSIGHT Q8: cost (modelled with an assumed rate card).**
> - Freight cost is **not in the data**. It is estimated as distance × an assumed INR/km rate by vehicle class, divided by vehicle capacity. The rates are a transparent, editable assumption.
> - **North** (Haryana, Rajasthan, Delhi, HP) costs about **₹5,600 per tonne**, 5× the South (about ₹1,100). It takes **34% of modelled spend** and has the **worst OTIF (11.8%)**. That is the most expensive *and* least reliable region.
> - Cost per tonne-km is roughly flat by design, so regional cost differences are distance-driven. The lever is **network design** (where stock sits), not haggling on rates.

## Part XIII: Q9, Breaking Points: High Volume, Low Success

In [24]:
st = (df.groupby("DestinationState").agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"), Carriers=("Carrier", "nunique"),
        TopCarrierShare=("Carrier", lambda s: s.value_counts(normalize=True).iloc[0]), MedianKm=("DistanceKm", "median"),
        Feasible=("SLAFeasible", "mean"), Lat=("DestLat", "median"), Lon=("DestLon", "median"))
        .query("Trips >= 20"))
vol_cut, ot_cut = st.Trips.median(), df.IsOnTime.mean()
st["Quadrant"] = np.select([(st.Trips >= vol_cut) & (st.OTIF < ot_cut), (st.Trips >= vol_cut)], ["Breaking point", "Core - healthy"],
                           np.where(st.OTIF < ot_cut, "Low volume - weak", "Low volume - healthy"))
st["Prescription"] = np.where(st.Quadrant != "Breaking point", "-",
                              np.where(st.Feasible < 0.5, "Fix SLA / consider regional hub (long-haul, infeasible plans)",
                                       np.where(st.TopCarrierShare > 0.4, "Change / dual-source carrier", "Hub + carrier mix review")))
st = st.sort_values("Trips", ascending=False)
st.to_csv(OUTPUTS / "state_breaking_points.csv")
display(st)
fig = px.scatter(st.reset_index(), x="Trips", y="OTIF", size="Trips", color="Quadrant", text="DestinationState", log_x=True,
                 title="Destination states: volume vs. OTIF (breaking points = high volume, below-network OTIF)")
fig.add_hline(y=ot_cut, line_dash="dot"); fig.add_vline(x=vol_cut, line_dash="dot")
fig.update_traces(textposition="top center"); fig.update_layout(yaxis_tickformat=".0%"); fig.show()

,Trips,OTIF,Carriers,TopCarrierShare,MedianKm,Feasible,Lat,Lon,Quadrant,Prescription
DestinationState,,,,,,,,,,
Tamil Nadu,1240,0.551,85,0.104,560.000,0.549,12.839,79.954,Core - healthy,-
Haryana,372,0.129,42,0.261,"2,240.000",0.212,28.374,76.835,Breaking point,"Fix SLA / consider regional hub (long-haul, in..."
Gujarat,329,0.435,27,0.191,"1,000.000",0.559,22.962,72.094,Core - healthy,-
Maharashtra,277,0.289,37,0.310,"1,199.000",0.227,18.751,73.877,Breaking point,"Fix SLA / consider regional hub (long-haul, in..."
Karnataka,257,0.603,29,0.362,110.000,0.868,12.751,77.763,Core - healthy,-
West Bengal,175,0.303,8,0.566,214.000,0.903,22.952,88.457,Breaking point,Change / dual-source carrier
Rajasthan,126,0.206,27,0.167,"1,178.000",0.278,27.236,75.917,Breaking point,"Fix SLA / consider regional hub (long-haul, in..."
Odisha,123,0.171,8,0.634,109.000,0.886,22.200,84.679,Breaking point,Change / dual-source carrier
Telangana,113,0.142,11,0.389,65.000,0.805,17.556,78.208,Low volume - weak,-


In [25]:
# Geospatial view: where are the delays? (scatter_geo works offline - no map tiles needed)
geo = df.groupby(["DestinationCity", "DestinationState"]).agg(Trips=("BookingID", "count"), OTIF=("IsOnTime", "mean"),
                                                               Lat=("DestLat", "median"), Lon=("DestLon", "median")).reset_index()
fig = px.scatter_geo(geo, lat="Lat", lon="Lon", size="Trips", color="OTIF", color_continuous_scale="RdYlGn", range_color=[0, 1],
                     hover_name="DestinationCity", hover_data=["DestinationState", "Trips"], size_max=40,
                     title="Delivery hot-spots: destination cities sized by trips, coloured by OTIF")
fig.update_geos(fitbounds="locations", showcountries=True, countrycolor="#94A3B8", showland=True, landcolor="#F8FAFC")
fig.show()

> **INSIGHT Q9: breaking points (high volume, below-network OTIF), and what to do about each.**
>
> | State | Trips | OTIF | Diagnosis | Prescription |
> |---|---|---|---|---|
> | **Haryana** | 372 | 12.9% | 2,240 km median; only 21% of plans feasible | **Regional hub / cross-dock in the North** plus a distance-based SLA |
> | **Maharashtra** | 277 | 28.9% | long-haul, 23% feasible | Distance-based SLA; West consolidation |
> | **Rajasthan** | 126 | 20.6% | long-haul, 28% feasible | Serve from the North hub |
> | **West Bengal** | 175 | 30.3% | short-haul, but one carrier has 57% of trips | **Change / dual-source carrier** |
> | **Odisha** | 123 | 17.1% | 109 km, one carrier has 63% of trips | **Change / dual-source carrier**; fix site dwell |
>
> **Rule of thumb:** a feasible plan with poor results points to the carrier. An infeasible plan on long distance points to the SLA and network (hub).

## Part XIV: Q10, Rule-Based Delay Early-Warning (at Booking Time)

Only information known **when the booking is made** is used: distance, planned transit time (and therefore planned speed / SLA feasibility), customer, carrier history, vehicle class and shipment type. The rules are fitted on bookings **before 1 Aug 2020** and tested on **August 2020** (an out-of-time test).

In [26]:
train = df[df.BookingDate < "2020-08-01"].copy()
test = df[df.BookingDate >= "2020-08-01"].copy()
carrier_hist = train.groupby("Carrier").IsOnTime.agg(["mean", "count"])
weak_carriers = set(carrier_hist.query("count >= 15 and mean < 0.25").index)
cust_hist = train.groupby("Customer").IsOnTime.agg(["mean", "count"])
weak_customers = set(cust_hist.query("count >= 15 and mean < 0.25").index)

RULES = {
    "R1 Infeasible SLA (planned speed > 10.4 km/h)": lambda d: d.SLAFeasible == 0,
    "R2 Long haul (> 1,200 km)": lambda d: d.DistanceKm > 1200,
    "R3 Carrier with weak history (< 25% on-time)": lambda d: d.Carrier.isin(weak_carriers),
    "R4 Customer lane with weak history (< 25% on-time)": lambda d: d.Customer.isin(weak_customers),
}
WEIGHTS = {"R1 Infeasible SLA (planned speed > 10.4 km/h)": 3, "R2 Long haul (> 1,200 km)": 1,
           "R3 Carrier with weak history (< 25% on-time)": 2, "R4 Customer lane with weak history (< 25% on-time)": 1}
def score(d):
    s = pd.Series(0, index=d.index)
    for name, rule in RULES.items():
        s += rule(d).astype(int) * WEIGHTS[name]
    return s

rule_stats = []
for name, rule in RULES.items():
    hit = rule(test)
    rule_stats.append({"Rule": name, "Weight": WEIGHTS[name], "Trips flagged (test)": int(hit.sum()),
                       "Late rate when flagged": 1 - test.loc[hit, "IsOnTime"].mean(), "Late rate otherwise": 1 - test.loc[~hit, "IsOnTime"].mean()})
display(pd.DataFrame(rule_stats))

for d in (train, test, df):
    d["RiskScore"] = score(d)
df["RiskBand"] = pd.cut(df.RiskScore, [-1, 0, 2, 4, 10], labels=["Low (0)", "Medium (1-2)", "High (3-4)", "Critical (5+)"])
test["RiskBand"] = pd.cut(test.RiskScore, [-1, 0, 2, 4, 10], labels=["Low (0)", "Medium (1-2)", "High (3-4)", "Critical (5+)"])
display(test.groupby("RiskBand").agg(Trips=("BookingID", "count"), LateRate=("IsOnTime", lambda s: 1 - s.mean())))

y_true = 1 - test.IsOnTime
for thr in (2, 3, 4):
    y_pred = (test.RiskScore >= thr).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum()); fp = int(((y_pred == 1) & (y_true == 0)).sum()); fn = int(((y_pred == 0) & (y_true == 1)).sum())
    acc = ((y_pred == y_true).mean())
    print(f"Alert if score >= {thr}: precision {tp/(tp+fp):.0%}, recall {tp/(tp+fn):.0%}, accuracy {acc:.0%}, alerts {y_pred.mean():.0%} of trips")
print("Baseline accuracy (always predict 'on time'):", f"{test.IsOnTime.mean():.0%}")

,Rule,Weight,Trips flagged (test),Late rate when flagged,Late rate otherwise
0,R1 Infeasible SLA (planned speed > 10.4 km/h),3,456,0.719,0.144
1,"R2 Long haul (> 1,200 km)",1,332,0.705,0.213
2,R3 Carrier with weak history (< 25% on-time),2,360,0.708,0.199
3,R4 Customer lane with weak history (< 25% on-t...,1,614,0.588,0.132


,Trips,LateRate
RiskBand,,
Low (0),678,0.060
Medium (1-2),226,0.199
High (3-4),197,0.655
Critical (5+),334,0.760


Alert if score >= 2: precision 69%, recall 83%, accuracy 82%, alerts 39% of trips
Alert if score >= 3: precision 72%, recall 82%, accuracy 84%, alerts 37% of trips
Alert if score >= 4: precision 72%, recall 64%, accuracy 80%, alerts 29% of trips
Baseline accuracy (always predict 'on time'): 67%


> **INSIGHT Q10: a 4-rule early-warning score is accurate enough to act on.**
> - **Rules, all known at booking:**
>   - Infeasible SLA (planned speed > 10.4 km/h): **+3**
>   - Carrier with weak history: **+2**
>   - Long haul (> 1,200 km): **+1**
>   - Customer lane with weak history: **+1**
> - **Out-of-time test (Aug-2020).** At score ≥ 3:
>   - **Precision 72%, recall 82%**
>   - **Accuracy 84%, vs 67% for "always on time"**
>   - 37% of trips get an alert
> - Late rate by risk band: **Low 6%, Medium 20%, High 66%, Critical 76%**.
> - **How customer service uses it:**
>   - **Critical:** call the customer at booking with a realistic ETA.
>   - **High:** dispatcher checks carrier and slot before release.
>   - **Low:** no action.

## Part XV: Scenario: What Would Lift OTIF by ≥ 5 pp Next Quarter?

In [27]:
base = df.IsOnTime.mean()
# Lever 1: distance-based SLA. Re-plan ETA = max(current plan, distance / 250 km per day) and re-score on-time
lev = df.dropna(subset=["LeadTimeHrs"]).copy()
lev["NewPlanHrs"] = np.maximum(lev.PlannedTransitHrs, lev.RealisticTransitHrs)
lev["OnTimeNewSLA"] = (lev.LeadTimeHrs <= lev.NewPlanHrs).astype(int)
lever1 = lev.OnTimeNewSLA.mean() - lev.IsOnTime.mean()
# Lever 2: move 'Exit' carriers' volume to the expected on-time level of their lanes
exit_c = car.query("Verdict == 'Exit / re-tender'").index   # 'Unknown Carrier' is a data gap, not a vendor, so it is excluded
ex = df[df.Carrier.isin(exit_c)]
lever2 = (ex.ExpectedOnTime.sum() - ex.IsOnTime.sum()) / len(df)
# Lever 3: early-warning on High/Critical bookings - conservative assumption: dispatch recovers 1 in 10 flagged late trips
flag = df.RiskScore >= 3
lever3 = 0.10 * ((flag) & (df.IsOnTime == 0)).sum() / len(df)
scen = pd.DataFrame({"Lever": ["1. Distance-based SLA (250 km/day rule)", "2. Re-tender / exit weakest carriers", "3. Booking-time early warning (10% recovery)"],
                     "OTIF uplift (pp)": [lever1 * 100, lever2 * 100, lever3 * 100]})
display(scen)
print(f"Baseline OTIF {base:.1%}; levers 2+3 alone (no SLA change): +{(lever2+lever3)*100:.1f} pp; all three: up to {base + lever1 + lever2 + lever3:.1%}")

,Lever,OTIF uplift (pp)
0,1. Distance-based SLA (250 km/day rule),15.562
1,2. Re-tender / exit weakest carriers,6.848
2,3. Booking-time early warning (10% recovery),4.969


Baseline OTIF 38.6%; levers 2+3 alone (no SLA change): +11.8 pp; all three: up to 65.9%


> **What lifts OTIF by ≥ 5 pp next quarter? Three levers (not additive; each measured against today's baseline).**
>
> | Lever | OTIF uplift | Note |
> |---|---|---|
> | **1. Distance-based SLA** (ETA = max(current plan, distance ÷ 250 km/day)) | **+15.6 pp** | Re-sets the promise to what trucks can actually do: more honest, and it stops penalising long-haul carriers |
> | **2. Re-tender the 12 weakest carriers** (volume moves to lane-expected performance) | **+6.8 pp** | About 800 trips; reallocate to the "preferred" list |
> | **3. Booking-time early warning** (recover 1 in 10 flagged late trips) | **+5.0 pp** | Conservative assumption; dispatch + CS intervention |
>
> **Even without touching the SLA, levers 2 + 3 deliver about +11.8 pp, more than double the ≥ 5 pp target.**

## Part XVI: Export Star Schema for Power BI

In [28]:
df["SLAStatus"] = np.where(df.SLAFeasible == 1, "Feasible plan", "Infeasible plan")
df["OnTimeStatus"] = np.where(df.IsOnTime == 1, "On time", "Late")
df["DistanceBandNo"] = df.DistanceBand.cat.codes + 1
df["RiskBandNo"] = df.RiskBand.cat.codes + 1
df["CarrierKey"] = df.Carrier.astype("category").cat.codes + 1
df["CustomerKey"] = df.Customer.astype("category").cat.codes + 1
fact = df[["BookingID", "BookingDay", "BookingDate", "PlannedETA", "ActualETA", "ShipmentType", "GpsProvider", "VehicleReg",
           "CarrierKey", "CustomerKey", "VehicleClass", "VehicleType", "CapacityUsedMT", "MaterialCategory", "Material", "IsEmptyReturn",
           "Origin", "OriginCity", "OriginState", "OriginZone", "Destination", "DestinationCity", "DestinationState", "DestinationZone",
           "Route", "Lane", "OriginLat", "OriginLon", "DestLat", "DestLon", "CurrentLat", "CurrentLon",
           "DistanceKm", "DistanceImputed", "DistanceBand", "DistanceBandNo", "CrowKm", "RouteToCrowRatio", "AnomalyType",
           "IsOnTime", "OnTimeStatus", "SLAStatus", "RiskBandNo", "PlannedTransitHrs", "RealisticTransitHrs", "PlannedSpeedKmh", "SLAFeasible", "LeadTimeHrs", "LeadTimeDays",
           "DelayHrs", "ValidLeadTime", "ExpectedOnTime", "RiskScore", "RiskBand", "EstFreightCostINR", "EstCostPerTonneINR", "RateInrPerKm"]].copy()
fact = fact.rename(columns={"BookingDay": "BookingDay", "BookingDate": "BookingDateTime"})
dim_carrier = (df.groupby(["CarrierKey", "Carrier"]).size().rename("Trips").reset_index()
               .merge(car[["Verdict", "VsExpected_pp"]].reset_index(), on="Carrier", how="left").fillna({"Verdict": "Not rated (< 30 trips)"}))
dim_customer = df.groupby(["CustomerKey", "Customer"]).size().rename("Trips").reset_index()
dim_zone = pd.DataFrame({"DestinationZone": sorted(df.DestinationZone.unique())})
dates = pd.date_range(df.BookingDay.min(), df.BookingDay.max(), freq="D")
dim_date = pd.DataFrame({"Date": dates})
dim_date["Year"] = dim_date.Date.dt.year
dim_date["MonthNo"] = dim_date.Date.dt.month
dim_date["MonthName"] = dim_date.Date.dt.strftime("%b")
dim_date["YearMonth"] = dim_date.Date.dt.strftime("%Y-%m")
dim_date["MonthStart"] = dim_date.Date.dt.to_period("M").dt.to_timestamp()
dim_date["WeekdayNo"] = dim_date.Date.dt.dayofweek + 1
dim_date["Weekday"] = dim_date.Date.dt.day_name()
dim_date["IsWeekend"] = (dim_date.WeekdayNo >= 6).astype(int)
dim_date["IsCorePeriod"] = ((dim_date.Date >= "2019-12-01") & (dim_date.Date < "2020-09-01")).astype(int)

exports = {"fact_trips.csv": fact, "dim_carrier.csv": dim_carrier, "dim_customer.csv": dim_customer,
           "dim_zone.csv": dim_zone, "dim_date.csv": dim_date}
for fname, frame in exports.items():
    out = frame.copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].dt.strftime("%Y-%m-%d %H:%M:%S") if c in ("BookingDateTime", "PlannedETA", "ActualETA") else out[c].dt.strftime("%Y-%m-%d")
    out.to_csv(PROCESSED / fname, index=False, encoding="utf-8")
    print(f"{fname:<18} {out.shape[0]:>6,} rows  {out.shape[1]:>2} cols")

fact_trips.csv      3,582 rows  57 cols
dim_carrier.csv       192 rows   5 cols
dim_customer.csv       34 rows   3 cols
dim_zone.csv            7 rows   1 cols
dim_date.csv          599 rows  10 cols


In [29]:
metrics = {
    "kpi": {k: float(v) for k, v in kpi.items()},
    "road_factor": ROAD_FACTOR,
    "monthly": m.assign(BookingMonth=m.BookingMonth.dt.strftime("%Y-%m")).to_dict(orient="records"),
    "zone": zone.reset_index().to_dict(orient="records"),
    "routes": routes.reset_index().head(12).to_dict(orient="records"),
    "band": band.reset_index().astype({"DistanceBand": str}).to_dict(orient="records"),
    "feasible": feas.reset_index().to_dict(orient="records"),
    "carrier": car.reset_index().to_dict(orient="records"),
    "material": mat.reset_index().to_dict(orient="records"),
    "material_test": {"chi2": float(chi2), "p": float(p), "cramers_v": float(cramers_v), "sla_cramers_v": float(v2)},
    "util": {k: float(v) for k, v in util.items()},
    "vehicle": vc.reset_index().to_dict(orient="records"),
    "anomaly": anom.reset_index().rename(columns={"index": "AnomalyType"}).to_dict(orient="records"),
    "weekday": wd.reset_index().to_dict(orient="records"),
    "cost": cost.reset_index().to_dict(orient="records"),
    "states": st.reset_index().to_dict(orient="records"),
    "rules": pd.DataFrame(rule_stats).to_dict(orient="records"),
    "risk_bands_test": test.groupby("RiskBand").agg(Trips=("BookingID", "count"), LateRate=("IsOnTime", lambda s: 1 - s.mean())).reset_index().astype({"RiskBand": str}).to_dict(orient="records"),
    "scenario": scen.to_dict(orient="records"),
    "base_otif": float(base),
}
txt = json.dumps(metrics, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
(OUTPUTS / "metrics.json").write_text(txt.replace("NaN", "null"))
print("metrics.json written")

metrics.json written


## Part XVII: Summary of Insights & Recommendations

| # | Question | Answer |
|---|---|---|
| 1 | OTIF health | **38.6%** network OTIF. The rise from 23% to 67% (Jun → Aug 2020) is mostly a shift to short-haul South lanes. |
| 2 | Bottlenecks | Long-haul (> 1,200 km) runs on a **flat ~100 h SLA that is never feasible** (OTIF 12–19%). Short-haul suffers from **site dwell time** (UP, Odisha: 6+ days for < 70 km). |
| 3 | Carriers | Rank against the lane: **12 carriers are ≥ 15 pp below expected → re-tender**. Ekta and Trans Cargo are *not* the problem. 217 trips have no carrier recorded. |
| 4 | Material | Statistically significant but weak (V = 0.12). The lane and SLA matter 3× more. |
| 5 | Utilisation | **10.6% of trips run empty-return**; 55% of heavy-trailer trips are < 100 km shuttles. |
| 6 | Anomalies | **24% of trips** have impossible, detour or round-trip GPS patterns, which corrupt ETAs. |
| 7 | Weekday | The Wednesday effect is not real; **weekend long-haul releases** are the weak spot. |
| 8 | Cost | The North costs about 5× the South per tonne, takes 34% of spend and has the worst OTIF. |
| 9 | Breaking points | **Haryana, Maharashtra, Rajasthan** → SLA + North hub; **West Bengal, Odisha** → change / dual-source carrier. |
| 10 | Early warning | A 4-rule score: precision 72% / recall 82% on an out-of-time test. |

### Three recommendations for next quarter (target: ≥ +5 pp OTIF)
1. **Re-tender the 12 weakest carriers** and shift volume to the 9 "preferred" carriers. *+6.8 pp.*
2. **Deploy the booking-time early-warning score** in the TMS / CS workflow. *+5.0 pp at a 10% recovery rate.*
3. **Replace the flat 100-hour ETA with a distance-based SLA** (250 km/day + buffer) and fix the 24% of GPS master-data anomalies. *Re-baselines the promise (+15.6 pp measured OTIF) and protects customer trust.*

*Next step (network):* a feasibility study for a **North cross-dock (Gurgaon / Haryana)** to serve Haryana, Rajasthan, Delhi and HP. That region is the costliest and least reliable.